### [Docker, Spark, and Iceberg: The Fastest Way to Try Iceberg!](https://tabular.io/blog/docker-spark-and-iceberg/)

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Jupyter").getOrCreate()
spark

## Load One Month of NYC Taxi/Limousine Trip Data

For this notebook, we will use the New York City Taxi and Limousine Commision Trip Record Data that's available on the AWS Open Data Registry. This contains data of trips taken by taxis and for-hire vehicles in New York City. We'll save this into an iceberg table called `taxis`.

To be able to rerun the notebook several times, let's drop the table if it exists to start fresh.

In [ ]:
# Create database if not exists
spark.sql("CREATE DATABASE IF NOT EXISTS nyc")

In [ ]:
# Drop table if exists
spark.sql("DROP TABLE IF EXISTS nyc.taxis")

In [ ]:
# Read one month's parquet from the mounted data path and save as an Iceberg table
df = spark.read.parquet("/home/iceberg/data/yellow_tripdata_2021-04.parquet")
df.write.saveAsTable("nyc.taxis")

In [ ]:
# Describe table metadata
spark.sql("DESCRIBE EXTENDED nyc.taxis").show(truncate=False)

In [ ]:
# Row count
spark.sql("SELECT COUNT(*) as cnt FROM nyc.taxis").show()

## Schema Evolution

Adding, dropping, renaming, or altering columns is easy and safe in Iceberg. In this example, we'll rename `fare_amount` to `fare` and `trip_distance` to `distance`. We'll also add a float column `fare_per_distance_unit` immediately after `distance`.

In [ ]:
spark.sql("ALTER TABLE nyc.taxis RENAME COLUMN fare_amount TO fare")

In [ ]:
spark.sql("ALTER TABLE nyc.taxis RENAME COLUMN trip_distance TO distance")

In [ ]:
spark.sql("ALTER TABLE nyc.taxis ALTER COLUMN distance COMMENT 'The elapsed trip distance in miles reported by the taximeter.'")

In [ ]:
spark.sql("ALTER TABLE nyc.taxis ALTER COLUMN distance TYPE double")

In [ ]:
# Reorder column (note: some catalog implementations may ignore AFTER)
spark.sql("ALTER TABLE nyc.taxis ALTER COLUMN distance AFTER fare")

In [ ]:
spark.sql("ALTER TABLE nyc.taxis ADD COLUMN fare_per_distance_unit float AFTER distance")

Let's update the new `fare_per_distance_unit` to equal `fare` divided by `distance`.

In [ ]:
spark.sql("UPDATE nyc.taxis SET fare_per_distance_unit = fare/distance")

In [ ]:
spark.sql("SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, fare, distance, fare_per_distance_unit FROM nyc.taxis").show(20, False)

## Expressive SQL for Row Level Changes
With Iceberg tables, `DELETE` queries can be used to perform row-level deletes. This is as simple as providing the table name and a `WHERE` predicate. If the filter matches an entire partition of the table, Iceberg will intelligently perform a metadata-only operation where it simply deletes the metadata for that partition.

Let's perform a row-level delete for all rows that have a `fare_per_distance_unit` greater than 4 or a `distance` greater than 2.0. This should leave us with relatively short trips that have a relatively high fare per distance traveled.

In [ ]:
spark.sql("DELETE FROM nyc.taxis WHERE fare_per_distance_unit > 4.0 OR distance > 2.0")

There are some fares that have a `null` for `fare_per_distance_unit` due to the distance being `0`. Let's remove those as well.

In [ ]:
spark.sql("DELETE FROM nyc.taxis WHERE fare_per_distance_unit is null")

In [ ]:
spark.sql("SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, fare, distance, fare_per_distance_unit FROM nyc.taxis").show(20, False)

In [ ]:
spark.sql("SELECT COUNT(*) as cnt FROM nyc.taxis").show()

## Partitioning

A table’s partitioning can be updated in place and applied only to newly written data. Query plans are then split, using the old partition scheme for data written before the partition scheme was changed, and using the new partition scheme for data written after. People querying the table don’t even have to be aware of this split. Simple predicates in WHERE clauses are automatically converted to partition filters that prune out files with no matches. This is what’s referred to in Iceberg as *Hidden Partitioning*.

In [ ]:
spark.sql("ALTER TABLE nyc.taxis ADD PARTITION FIELD VendorID")

## Metadata Tables

Iceberg tables contain very rich metadata that can be easily queried. For example, you can retrieve the manifest list for any snapshot, simply by querying the table's `snapshots` table.

In [ ]:
spark.sql("SELECT snapshot_id, manifest_list FROM nyc.taxis.snapshots").show(50, False)

The `files` table contains loads of information on data files, including column level statistics such as null counts, lower bounds, and upper bounds.

In [ ]:
spark.sql("SELECT file_path, file_format, record_count, null_value_counts, lower_bounds, upper_bounds FROM nyc.taxis.files").show(50, False)

## Time Travel

The history table lists all snapshots and which parent snapshot they derive from. The `is_current_ancestor` flag let's you know if a snapshot is part of the linear history of the current snapshot of the table.

In [ ]:
spark.sql("SELECT * FROM nyc.taxis.history").show(50, False)

You can time-travel by altering the `current-snapshot-id` property of the table to reference any snapshot in the table's history. Let's revert the table to it's original state by traveling to the very first snapshot ID.

In [ ]:
# Load history into a DataFrame
df = spark.sql("SELECT * FROM nyc.taxis.history")
df.show(5, False)

In [ ]:
original_snapshot = df.first()['snapshot_id']
spark.sql(f"CALL system.rollback_to_snapshot('nyc.taxis', {original_snapshot})")
original_snapshot

In [ ]:
spark.sql("SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, fare, distance, fare_per_distance_unit FROM nyc.taxis").show(20, False)

Another look at the history table shows that the original state of the table has been added as a new entry
with the original snapshot ID.

In [ ]:
spark.sql("SELECT * FROM nyc.taxis.history").show(50, False)

In [ ]:
spark.sql("SELECT COUNT(*) as cnt FROM nyc.taxis").show()